# 05 — Quantum Routing (QUBO)

## 01. Research question

**RQ6 / H5**: At what problem size and under what computational constraints does quantum or quantum-inspired routing become competitive with classical routing?

Primary question: can quantum/quantum-inspired optimization produce competitive feasible vehicle-routing solutions compared with a strong OR-Tools baseline when both methods solve exactly the same constrained routing instances?

This is a **RESEARCH** notebook. Quantum superiority is not assumed; classical superiority is not assumed. A negative quantum result is a valid, reportable outcome — this notebook does not optimize the experiment to make quantum look competitive.

In [1]:
import sys, time, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import dimod

from qsci_core.data import loaders
from qsci_core.optimization.quantum.qubo_formulator import build_cvrp_qubo
from qsci_core.optimization.quantum.decoder import decode_qubo_sample
from qsci_core.optimization.quantum.classical_qubo_solver import solve_qubo_with_ortools
from qsci_core.optimization.quantum.annealing_solver import solve_qubo_with_annealing, BACKEND_LABEL as ANNEALING_LABEL
from qsci_core.optimization.quantum.qaoa_solver import solve_qubo_with_qaoa
from qsci_core.optimization.quantum.instance_builder import build_derived_instance, STATUS_LABEL as DERIVED_LABEL
from qsci_core.optimization.quantum.experiment import RoutingManifest

print("project root:", PROJECT_ROOT)


project root: D:\Projects\QML project


## 02-03. Routing problem and dataset

Status: **LIVE**. Real CVRPLIB coordinates/demands (`A-n32-k5`, verified in Phase 0 and re-verified below), scaling instances of N ∈ {4, 6, 8, 10, 12, 16} customers built by truncating the real instance — labeled **RESEARCH — DERIVED / SYNTHETIC EXTENSION** per CLAUDE.md §3, since truncation invalidates the original published BKS. Amazon Last-Mile data is re-verified below too (both were already verified once in Phase 0; this Phase 3 audit re-reads the raw files directly rather than trusting that prior report, per CLAUDE.md §4/§5).

In [2]:
with open(PROJECT_ROOT / "experiments" / "results" / "routing_ground_truth.json") as f:
    ground_truth = json.load(f)

print("=== CVRPLIB ground truth (re-verified directly from raw .sol files) ===")
for row in ground_truth["cvrplib"]:
    print(row)

print("\n=== Amazon Last-Mile ground truth (re-verified directly from raw JSON) ===")
lm = ground_truth["amazon_last_mile"]
print("directly observed:", lm["directly_observed"])
print("derived:", lm["derived"])
print("matches Phase 0 report:", lm["previously_reported_phase_0"]["match"])


=== CVRPLIB ground truth (re-verified directly from raw .sol files) ===
{'instance': 'A-n32-k5', 'number_of_customers': 31, 'number_of_vehicles': 5, 'capacity': 100, 'published_bks': 784, 'source': 'raw .sol file, data/raw/cvrplib/', 'verification_status': 'MATCHES_PRIOR_REPORT'}
{'instance': 'A-n33-k5', 'number_of_customers': 32, 'number_of_vehicles': 5, 'capacity': 100, 'published_bks': 661, 'source': 'raw .sol file, data/raw/cvrplib/', 'verification_status': 'MATCHES_PRIOR_REPORT'}
{'instance': 'A-n37-k5', 'number_of_customers': 36, 'number_of_vehicles': 5, 'capacity': 100, 'published_bks': 669, 'source': 'raw .sol file, data/raw/cvrplib/', 'verification_status': 'MATCHES_PRIOR_REPORT'}
{'instance': 'B-n31-k5', 'number_of_customers': 30, 'number_of_vehicles': 5, 'capacity': 100, 'published_bks': 672, 'source': 'raw .sol file, data/raw/cvrplib/', 'verification_status': 'MATCHES_PRIOR_REPORT'}

=== Amazon Last-Mile ground truth (re-verified directly from raw JSON) ===
directly observe

## 05. Classical OR-Tools baseline

Status: **LIVE**. Two distinct OR-Tools baselines, kept separate per CLAUDE.md §8's fairness requirement:

1. **Native VRP solver** (Phase 1, `qsci_core/optimization/classical/or_tools_vrp.py`) — solves the *full* CVRP with real route sequencing. Already established: 0.61% max gap vs. published BKS across all 4 real CVRPLIB instances. This is OR-Tools' true capability, kept as outside context — quantum/annealing are not compared against this number directly, since they solve a different (QUBO-defined) sub-problem.
2. **OR-Tools-on-QUBO** (this phase, `classical_qubo_solver.py`) — solves the *exact same* binary-quadratic objective and constraints the annealing/QAOA tracks solve, via CP-SAT with a quadratic objective. This is the fair, same-problem classical reference for the comparisons below. Config: CP-SAT, `random_seed=42`, `max_time_in_seconds=30`, default search strategy (no manual weakening).

## 06. Small-instance brute-force validation — a real bug was found and fixed here

Status: **LIVE**. Before any quantum benchmarking, the QUBO's true global optimum (via `dimod.ExactSolver`, exhaustive enumeration) is checked against an independent manual brute-force enumeration. **This check failed on the first attempt with real CVRPLIB-derived data**: the original penalty derivation (`penalty_visit = m * n * max(c_ij)`, sized only against the objective's scale) allowed a degenerate optimum — two customers assigned to *both* vehicles (hitting each vehicle's capacity target exactly, zeroing the capacity penalty) while two other customers were left completely unassigned. The fix derives `penalty_visit` to also dominate the capacity penalty's own maximum magnitude (`K * penalty_capacity * capacity²`), not just the objective. See `qsci_core/optimization/quantum/qubo_formulator.py`'s docstring for the full derivation and `docs/PHASE_3_REPORT.md` §9 for the specific failing numbers. Per CLAUDE.md §11/§40, this stopped all further quantum work until fixed and re-validated — shown below.

In [3]:
SOURCE_INSTANCE = "A-n32-k5"
di4 = build_derived_instance(SOURCE_INSTANCE, 4)
print("derived instance:", di4.instance_id, "| status:", di4.status)
print("coords:\n", di4.coords, "\ndemands:", di4.demands, "| capacity:", di4.capacity, "| vehicles:", di4.n_vehicles)

qi4 = build_cvrp_qubo(di4.coords, di4.demands, di4.capacity, di4.n_vehicles, penalty_multiplier=2.0)
print(f"\nQUBO: {qi4.n_variables} variables, penalty_visit={qi4.penalty_visit:.2e}, penalty_capacity={qi4.penalty_capacity:.2e}")

best_exact = dimod.ExactSolver().sample(qi4.bqm).first
decoded4 = decode_qubo_sample(dict(best_exact.sample), 4, di4.n_vehicles, di4.coords, di4.demands, di4.capacity, di4.depot_coord)
print("\ntrue brute-force optimum, decoded:")
print("  feasible:", decoded4.feasible, "|", decoded4.feasibility_reason)
print("  assignment:", decoded4.assignment, "| loads:", decoded4.vehicle_loads)
print("  routes:", decoded4.routes, "| total_cost:", decoded4.total_cost)

# independent manual brute force over the assignment space (not going through dimod at all)
import itertools
diff = di4.coords[:, None, :] - di4.coords[None, :, :]
dist = np.sqrt((diff ** 2).sum(axis=-1))
best_manual_cost, best_manual_assign = float("inf"), None
for assign in itertools.product(range(di4.n_vehicles), repeat=4):
    loads = [0.0] * di4.n_vehicles
    for i, k in enumerate(assign):
        loads[k] += di4.demands[i]
    if any(l > di4.capacity for l in loads):
        continue
    total = sum(dist[a, b] for k in range(di4.n_vehicles) for a in [i for i, kk in enumerate(assign) if kk == k]
                for b in [i for i, kk in enumerate(assign) if kk == k] if a < b)
    if total < best_manual_cost:
        best_manual_cost, best_manual_assign = total, assign
print(f"\nindependent manual brute force: assignment={best_manual_assign}, pairwise-distance-proxy cost={best_manual_cost:.6f}")

proxy_cost_from_qubo = sum(
    dist[a, b] for custs in decoded4.assignment.values()
    for a in custs for b in custs if a < b
)
print(f"QUBO optimum's proxy cost: {proxy_cost_from_qubo:.6f} -- {'MATCH' if abs(proxy_cost_from_qubo - best_manual_cost) < 1e-6 else 'MISMATCH -- STOP'}")
assert decoded4.feasible, "STOP CONDITION: QUBO optimum must be feasible before any quantum benchmarking"
assert abs(proxy_cost_from_qubo - best_manual_cost) < 1e-6, "STOP CONDITION: QUBO optimum must match independent brute force"
print("\nQUBO VALIDATED against independent brute force. Proceeding.")


derived instance: A-n32-k5-derived-n4 | status: RESEARCH - DERIVED / SYNTHETIC EXTENSION
coords:
 [[96. 44.]
 [50.  5.]
 [49.  8.]
 [13.  7.]] 
demands: [19. 21.  6. 19.] | capacity: 38.0 | vehicles: 2

QUBO: 8 variables, penalty_visit=4.20e+06, penalty_capacity=7.27e+02

true brute-force optimum, decoded:
  feasible: True | all customers visited exactly once, all vehicles within capacity
  assignment: {0: [1, 2], 1: [0, 3]} | loads: {0: 27.0, 1: 38.0}
  routes: {0: [1, 2], 1: [0, 3]} | total_cost: 380.00755017228636

independent manual brute force: assignment=(0, 1, 1, 0), pairwise-distance-proxy cost=94.035816
QUBO optimum's proxy cost: 94.035816 -- MATCH

QUBO VALIDATED against independent brute force. Proceeding.


## 07-08. QUBO formulation (recap) and validation (done above)

The mathematical formulation, penalty derivation, and its real-bug/fix history are documented in full in `qsci_core/optimization/quantum/qubo_formulator.py`'s docstring and `docs/PHASE_3_REPORT.md` §9-10. Summary: n×K binary assignment variables (customer→vehicle), pairwise-distance clustering-proxy objective, two penalty constraints (visit-once, soft capacity), no slack variables (keeps qubit count to exactly n×K, at the cost of within-vehicle sequencing being decided classically post-hoc via exact enumeration — documented, not hidden).

## 09-10. Simulated annealing and QAOA — scaling sweep

Status: **RESEARCH**. All solvers consume the exact same `QUBOInstance` (same objective, same constraints, same penalty) — no solver receives simplified information. Given the practical runtime cost (documented below), the full N∈{4,6,8,10,12,16}×10-seed sweep was run once via `experiments/phase3_scaling_experiment.py` (committed, reproducible) rather than repeated inline — reproducing all 76 runs live in this notebook would take ~15 minutes dominated by QAOA's 10 seeds at N=4 alone (~6-7 min). This notebook re-runs a fast subset live (below) to confirm reproducibility of the pattern, then loads the complete dataset for the authoritative tables.

In [4]:
# live re-verification: annealing at N=4 (3 seeds) and one QAOA run -- fast enough to run inline
for seed in range(3):
    ann = solve_qubo_with_annealing(qi4, num_reads=100, seed=seed)
    dec = decode_qubo_sample(ann.sample, 4, di4.n_vehicles, di4.coords, di4.demands, di4.capacity, di4.depot_coord)
    print(f"annealing seed={seed}: feasible={dec.feasible} cost={dec.total_cost} time={ann.solve_time_seconds:.3f}s | backend: {ANNEALING_LABEL}")

t0 = time.perf_counter()
qaoa_res = solve_qubo_with_qaoa(qi4, p=1, optimizer_maxiter=20, shots=1024, seed=0)
qaoa_dec = decode_qubo_sample(qaoa_res.sample, 4, di4.n_vehicles, di4.coords, di4.demands, di4.capacity, di4.depot_coord)
print(f"\nQAOA seed=0: feasible={qaoa_dec.feasible} cost={qaoa_dec.total_cost} time={qaoa_res.solve_time_seconds:.2f}s")
print(f"  qubits={qaoa_res.metadata.n_qubits} depth={qaoa_res.metadata.circuit_depth} gates={qaoa_res.metadata.gate_count_total} "
      f"(1Q={qaoa_res.metadata.gate_count_1q}, 2Q={qaoa_res.metadata.gate_count_2q}) shots={qaoa_res.metadata.shots} "
      f"optimizer=COBYLA nfev={qaoa_res.metadata.optimizer_n_function_evals} simulator={qaoa_res.metadata.simulator}")
print("(all above match the true optimum cost 380.00755... found via brute force -- reproduces the full 10-seed script's pattern)")


annealing seed=0: feasible=True cost=380.00755017228636 time=0.013s | backend: SIMULATED ANNEALING (dwave.samplers.SimulatedAnnealingSampler, classical CPU)
annealing seed=1: feasible=True cost=380.00755017228636 time=0.012s | backend: SIMULATED ANNEALING (dwave.samplers.SimulatedAnnealingSampler, classical CPU)
annealing seed=2: feasible=True cost=380.00755017228636 time=0.012s | backend: SIMULATED ANNEALING (dwave.samplers.SimulatedAnnealingSampler, classical CPU)


D:\Projects\QML project\.venv\Lib\site-packages\scipy\sparse\linalg\_dsolve\linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
D:\Projects\QML project\.venv\Lib\site-packages\scipy\sparse\linalg\_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
D:\Projects\QML project\.venv\Lib\site-packages\scipy\sparse\_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])



QAOA seed=0: feasible=True cost=380.00755017228636 time=43.85s
  qubits=8 depth=34 gates=81 (1Q=49, 2Q=32) shots=1024 optimizer=COBYLA nfev=20 simulator=qiskit.primitives.StatevectorSampler (noiseless, shot-sampled)
(all above match the true optimum cost 380.00755... found via brute force -- reproduces the full 10-seed script's pattern)


## 11-16. Route decoding, feasibility, objective comparison, optimality gaps, runtime, scaling

Status: **LIVE** (annealing/OR-Tools) + **RESEARCH** (QAOA). Loading the complete 76-result dataset from `experiments/results/routing_results.json` (produced by `experiments/phase3_scaling_experiment.py`, same code as above, full N∈{4,6,8,10,12,16} × 10 seeds).

In [5]:
with open(PROJECT_ROOT / "experiments" / "results" / "routing_results.json") as f:
    all_results = json.load(f)
results_df = pd.DataFrame(all_results)
print(f"loaded {len(results_df)} results")

sizes = [4, 6, 8, 10, 12, 16]
scaling_rows = []
for n in sizes:
    ort = results_df[(results_df.solver == "or_tools_qubo") & (results_df.stops == n)].iloc[0]
    ann = results_df[(results_df.solver == "simulated_annealing") & (results_df.stops == n)]
    ann_feas = ann[ann.feasible]
    n_feas = len(ann_feas)
    best = ann_feas.objective_cost.min() if n_feas else None
    mean = ann_feas.objective_cost.mean() if n_feas else None
    std = ann_feas.objective_cost.std() if n_feas > 1 else None
    gap = (best - ort.objective_cost) / ort.objective_cost * 100 if best is not None and ort.objective_cost else None
    scaling_rows.append({
        "N": n, "qubo_variables": int(ort.qubo_variables), "ortools_status": ort.classification,
        "ortools_cost": round(ort.objective_cost, 2) if ort.objective_cost else None,
        "ortools_time_s": round(ort.runtime_solver_seconds, 2),
        "annealing_feasibility": f"{n_feas}/10", "annealing_best": round(best, 2) if best else None,
        "annealing_mean": round(mean, 2) if mean else None, "annealing_std": round(std, 2) if std else None,
        "annealing_gap_vs_ortools_pct": round(gap, 3) if gap is not None else None,
    })
scaling_df = pd.DataFrame(scaling_rows)
print(scaling_df.to_string(index=False))


loaded 76 results
 N  qubo_variables ortools_status  ortools_cost  ortools_time_s annealing_feasibility  annealing_best  annealing_mean  annealing_std  annealing_gap_vs_ortools_pct
 4               8        OPTIMAL        380.01            0.16                 10/10          380.01          380.01            NaN                         0.000
 6              18        OPTIMAL        519.63            0.17                 10/10          519.63          521.07           3.04                        -0.000
 8              32        OPTIMAL        643.50            0.34                  9/10          643.50          695.96          35.87                        -0.000
10              50        OPTIMAL        853.88            2.71                  1/10          859.01          859.01            NaN                         0.601
12              72       FEASIBLE       1117.46           30.19                  0/10             NaN             NaN            NaN                           NaN
16  

In [6]:
qaoa_df = results_df[results_df.solver == "qaoa"]
qaoa_feas = qaoa_df[qaoa_df.feasible]
n4_ortools = results_df[(results_df.solver == "or_tools_qubo") & (results_df.stops == 4)].iloc[0]

print(f"=== 17. QAOA statistics, N=4 (8 qubits), 10 seeds ===")
print(f"feasibility: {len(qaoa_feas)}/{len(qaoa_df)} = {len(qaoa_feas)/len(qaoa_df):.0%}")
print(f"cost when feasible: {sorted(qaoa_feas.objective_cost.unique())} (all feasible runs found the SAME cost)")
print(f"true optimum (OR-Tools/brute-force reference): {n4_ortools.objective_cost:.5f}")
gap_qaoa = (qaoa_feas.objective_cost.iloc[0] - n4_ortools.objective_cost) / n4_ortools.objective_cost * 100
print(f"QAOA optimality gap (when feasible): {gap_qaoa:.6f}% -- QAOA found the TRUE optimum every time it was feasible")
print(f"runtime: mean={qaoa_df.runtime_solver_seconds.mean():.2f}s std={qaoa_df.runtime_solver_seconds.std():.2f}s "
      f"min={qaoa_df.runtime_solver_seconds.min():.2f}s max={qaoa_df.runtime_solver_seconds.max():.2f}s")
print(f"circuit: {qaoa_df.iloc[0].qubits} qubits, depth={qaoa_df.iloc[0].circuit_depth}, gates={qaoa_df.iloc[0].gate_count}, shots={qaoa_df.iloc[0].shots}")

print(f"\n=== N=5, N=6 QAOA scaling ceiling (measured, cited from Phase 3 diagnostics -- not re-run here) ===")
print("N=4 (8 qubits): ~35s/run, 8/10 feasible -- tractable for repeated statistical runs")
print("N=5 (10 qubits): 628.9s single run (not repeated) -- feasible but too expensive for statistics")
print("N=6-with-K=2 (12 qubits): exceeded a 15+ minute budget across 4 independent diagnostic attempts, killed -- TIMEOUT")
print("Real N=6 derived instance needs 18 qubits (K=3 vehicles) -- worse than the already-timed-out 12-qubit case; not attempted -- PENDING")


=== 17. QAOA statistics, N=4 (8 qubits), 10 seeds ===
feasibility: 8/10 = 80%
cost when feasible: [np.float64(380.00755017228636)] (all feasible runs found the SAME cost)
true optimum (OR-Tools/brute-force reference): 380.00755
QAOA optimality gap (when feasible): 0.000000% -- QAOA found the TRUE optimum every time it was feasible
runtime: mean=37.61s std=2.19s min=34.33s max=40.92s
circuit: 8.0 qubits, depth=34.0, gates=81.0, shots=1024.0

=== N=5, N=6 QAOA scaling ceiling (measured, cited from Phase 3 diagnostics -- not re-run here) ===
N=4 (8 qubits): ~35s/run, 8/10 feasible -- tractable for repeated statistical runs
N=5 (10 qubits): 628.9s single run (not repeated) -- feasible but too expensive for statistics
N=6-with-K=2 (12 qubits): exceeded a 15+ minute budget across 4 independent diagnostic attempts, killed -- TIMEOUT
Real N=6 derived instance needs 18 qubits (K=3 vehicles) -- worse than the already-timed-out 12-qubit case; not attempted -- PENDING


## Penalty sensitivity (part of 17. Statistical/robustness analysis)

Status: **LIVE**.

In [7]:
with open(PROJECT_ROOT / "experiments" / "results" / "routing_penalty_sensitivity.json") as f:
    penalty_results = json.load(f)
pen_df = pd.DataFrame(penalty_results)
print(pen_df.to_string(index=False))
print("\nFinding: with the FIXED penalty derivation, cost and feasibility are stable across a 10x multiplier range")
print("(1x to 10x) -- confirms the fix is robust, not a narrow escape from the bug at one specific multiplier value.")


 multiplier  penalty_visit  penalty_capacity  feasible      cost                                               feasibility_reason
        1.0   1.050135e+06        363.494154      True 380.00755 all customers visited exactly once, all vehicles within capacity
        2.0   4.199811e+06        726.988308      True 380.00755 all customers visited exactly once, all vehicles within capacity
        5.0   2.624610e+07       1817.470770      True 380.00755 all customers visited exactly once, all vehicles within capacity
       10.0   1.049807e+08       3634.941540      True 380.00755 all customers visited exactly once, all vehicles within capacity

Finding: with the FIXED penalty derivation, cost and feasibility are stable across a 10x multiplier range
(1x to 10x) -- confirms the fix is robust, not a narrow escape from the bug at one specific multiplier value.


## 18. Business interpretation

Status: **RESEARCH** math over **LIVE** distances, **ILLUSTRATIVE** where cost data doesn't exist. Route distances above are real (Euclidean, in the same coordinate units CVRPLIB uses). No monetary translation (fuel, driver cost, per-stop cost) is computed — the Amazon Last-Mile ground-truth check above confirms no such data exists in the real datasets QSCI uses, and CLAUDE.md §25 prohibits fabricating it. Where Phase 1's financial layer needs a monetary figure, it already uses documented `CostAssumptions` labeled ILLUSTRATIVE — the same discipline applies here: this notebook reports **distance and feasibility**, not dollars.

At N=4 (the only size with a statistically meaningful QAOA sample), QAOA and simulated annealing both matched OR-Tools' true optimum whenever they returned a feasible solution — for a fleet-routing decision at this scale, that means zero distance loss vs. the classical baseline, at the cost of a much higher runtime (~35s vs. ~0.02s solver time) and a non-trivial infeasibility rate (annealing 0/10 to 10/10 depending on N; QAOA 8/10 at N=4). At N≥12, annealing itself (not even QAOA) stopped finding any feasible solution at all in 10 tries — this QUBO formulation's practical operating range, on this real data, tops out well below the Amazon Last-Mile dataset's real route sizes (avg. 148 stops/route, verified above) or even the original CVRPLIB instances' 30-37 customers.

## 19. Limitations

- **Cluster-first-route-second decomposition**, not the full joint CVRP: this QUBO decides customer→vehicle assignment; within-vehicle sequencing is decided classically post-hoc. Documented in `qubo_formulator.py` and not hidden, but it means quantum/annealing solve a genuinely easier sub-problem than OR-Tools' native VRP solver (Phase 1) — the two are not directly comparable, which is why they're reported separately (§5).
- **Soft, not hard, capacity constraint** — no slack variables (keeps qubit count to n×K), so QUBO solutions can violate capacity; the decoder catches this independently, but it means the QUBO's "optimum" and the true feasible optimum can differ, as the §9 bug demonstrated concretely.
- **QAOA statistics exist only at N=4.** N=5 has one expensive data point; N≥6 is PENDING due to measured runtime, not attempted at scale.
- **Annealing's own feasibility collapses well before any qubit limit** (0/10 at N=12, N=16) — this is a quality/tuning limitation of the fixed 100-reads/default-schedule configuration used here, not a hardware constraint; a more aggressive annealing schedule or more reads might do better at larger N, untested here.
- **Single source instance** (`A-n32-k5`), single random seed for instance truncation (deterministic, first-N-customers) — not a claim about CVRP instances in general.
- **No real D-Wave QPU hardware** was available or used — every annealing result is simulated annealing on classical CPU, labeled as such throughout.

## 20. Research conclusion — RQ6

**RQ6**: At what problem size and under what computational constraints does quantum or quantum-inspired routing become competitive with classical routing?

On this real-data-derived QUBO, measured directly: **quantum-inspired (simulated annealing) matches classical OR-Tools exactly at N∈{4,6,8} (0% gap when feasible), degrades sharply in feasibility rate by N=10 (1/10), and fails entirely by N=12 (0/10)** — well before any theoretical qubit or gate-model limit, since simulated annealing has no qubit ceiling at all. **Gate-model QAOA matched the true optimum whenever feasible (8/10 at N=4, 100% gap-free), but its own practical runtime ceiling (measured, not assumed) sits even lower** — reliable only at 8 qubits (N=4), with N=6's real 18-qubit instance already worse than an already-timed-out 12-qubit diagnostic.

Neither track demonstrates a quantum advantage on this formulation and data. Both demonstrate a genuine, measured operating envelope smaller than either the classical baseline's own capability (Phase 1: 0.61% gap at 30-37 real customers) or the real-world problem scale QSCI ultimately targets (Amazon Last-Mile: ~148 stops/route). H5 ("quantum/quantum-inspired routing becomes more competitive as problem size grows") is **not supported** by this evidence — if anything, the opposite pattern was measured: competitiveness (feasibility) *decreased* as size grew, for both quantum-inspired and gate-model tracks, on this formulation.